In [1]:
import requests

import numpy as np
import pandas as pd

import plotly.express as px
import folium

import os
from pathlib import Path
from urllib.request import urlretrieve
from urllib.error import HTTPError, URLError
from zipfile import ZipFile

c:\Users\Lil\miniconda3\envs\aca\Lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [5]:
weather_daily = pd.read_csv('../data/citibike/JC/jersey_weather_2025.csv')
citibike_df = pd.read_csv('../data/citibike/JC/JC2025_Enriched.csv')

In [6]:
weather_daily['date'] = pd.to_datetime(weather_daily['date'], errors="coerce")
citibike_df['date'] = pd.to_datetime(citibike_df['date'], errors="coerce")

In [7]:
citibike_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 999225 entries, 0 to 999224
Data columns (total 20 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   ride_id                999225 non-null  str           
 1   rideable_type          999225 non-null  str           
 2   started_at             999225 non-null  str           
 3   ended_at               999225 non-null  str           
 4   start_station_name     999224 non-null  str           
 5   start_station_id       999224 non-null  str           
 6   end_station_name       998439 non-null  str           
 7   end_station_id         998282 non-null  str           
 8   start_lat              999225 non-null  float64       
 9   start_lng              999225 non-null  float64       
 10  end_lat                999225 non-null  float64       
 11  end_lng                999225 non-null  float64       
 12  member_casual          999225 non-null  str           


Weather Time Series Visualization

In [8]:
monthly_rides = (citibike_df
                 .groupby('month',as_index=False)
                 .agg(number_of_rides = ('ride_id',"count"))
                 )
monthly_rides

,month,number_of_rides
0,2024-12,2
1,2025-01,50589
2,2025-02,45250
3,2025-03,73277
4,2025-04,81533
5,2025-05,93202
6,2025-06,96736
7,2025-07,107374
8,2025-08,108001
9,2025-09,115580


Number of rides per monhts

In [9]:

fig = px.bar(
    monthly_rides,
    x="month",
    y="number_of_rides",
    title="Number of Citi Bike Rides per Month"
)

fig.update_layout(
    xaxis_title="Month",
    yaxis_title="Number of Rides",
)

fig.show()

Number of Rides per Season

In [10]:
season_rides = (
    citibike_df
    .groupby("season", as_index=False)
    .agg(
        number_of_rides=("ride_id", "count")
    )
)

season_order = ["Winter", "Spring", "Summer", "Autumn"]

season_rides["season"] = pd.Categorical(
    season_rides["season"],
    categories=season_order,
    ordered=True
)

season_rides = season_rides.sort_values("season")

season_rides

,season,number_of_rides
3,Winter,143703
1,Spring,248012
2,Summer,312111
0,Autumn,295399


In [11]:
fig = px.bar(
    season_rides,
    x="season",
    y="number_of_rides",
    title="Number of Citi Bike Rides per Season",
    text_auto=True
)

fig.update_layout(
    xaxis_title="Season",
    yaxis_title="Number of Rides"
)

fig.show()

Top 10 Start Stations

In [12]:
top_start_stations = (
    citibike_df
    .dropna(subset=["start_station_name"])
    .groupby("start_station_name", as_index=False)
    .agg(
        number_of_departures=("ride_id", "count")
    )
    .sort_values("number_of_departures", ascending=False)
    .head(10)
)

top_start_stations

,start_station_name,number_of_departures
52,Grove St PATH,45004
58,Hoboken Terminal - Hudson St & Hudson Pl,25889
53,Hamilton Park,22259
95,River St & Newark St,21383
86,Newport PATH,20663
18,Bergen Ave & Sip Ave,20398
44,Exchange Pl,20008
0,11 St & Washington St,19481
94,River St & 1 St,19143
87,Newport Pkwy,18720


In [13]:
fig = px.bar(
    top_start_stations.sort_values("number_of_departures"),
    x="number_of_departures",
    y="start_station_name",
    orientation="h",
    title="Top 10 Start Stations by Number of Departures",
    text_auto=True
)

fig.update_layout(
    xaxis_title="Number of Departures",
    yaxis_title="Start Station"
)

fig.show()

Merge with Weather Data to See Patterns

In [14]:
daily_rides = (
    citibike_df
    .groupby("date", as_index=False)
    .agg(
        number_of_rides=("ride_id", "count")
    )
)
daily_rides["date"] = pd.to_datetime(daily_rides["date"])
daily_rides.head()

,date,number_of_rides
0,2024-12-31,2
1,2025-01-01,1179
2,2025-01-02,1710
3,2025-01-03,1770
4,2025-01-04,1337


In [15]:
weather_daily.head()

,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,snowfall_sum,wind_speed_10m_max,date
0,11.0,3.9,7.4,4.5,4.5,0.0,23.2,2025-01-01
1,5.4,0.3,2.6,0.0,0.0,0.0,25.1,2025-01-02
2,3.2,-1.9,0.4,0.0,0.0,0.0,17.1,2025-01-03
3,-0.1,-2.7,-1.4,0.0,0.0,0.0,26.1,2025-01-04
4,0.4,-3.6,-2.2,0.0,0.0,0.0,19.9,2025-01-05


In [16]:
bike_weather_daily = daily_rides.merge(
    weather_daily,
    on="date",
    how="left"
)

bike_weather_daily.head()

,date,number_of_rides,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,rain_sum,snowfall_sum,wind_speed_10m_max
0,2024-12-31,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025-01-01,1179,11.0,3.9,7.4,4.5,4.5,0.0,23.2
2,2025-01-02,1710,5.4,0.3,2.6,0.0,0.0,0.0,25.1
3,2025-01-03,1770,3.2,-1.9,0.4,0.0,0.0,0.0,17.1
4,2025-01-04,1337,-0.1,-2.7,-1.4,0.0,0.0,0.0,26.1


In [17]:
bike_weather_daily.isna().sum()

date                   0
number_of_rides        0
temperature_2m_max     1
temperature_2m_min     1
temperature_2m_mean    1
precipitation_sum      1
rain_sum               1
snowfall_sum           1
wind_speed_10m_max     1
dtype: int64

Rides and Average Temperature

In [18]:
fig = px.scatter(
    bike_weather_daily,
    x="temperature_2m_mean",
    y="number_of_rides",
    trendline="ols",
    title="Daily Rides vs Average Temperature"
)

fig.update_layout(
    xaxis_title="Average Daily Temperature",
    yaxis_title="Number of Rides"
)

fig.show()

Rides and Wind Speed

In [19]:
fig = px.scatter(
    bike_weather_daily,
    x="wind_speed_10m_max",
    y="number_of_rides",
    trendline="ols",
    title="Daily Rides vs Maximum Wind Speed"
)

fig.update_layout(
    xaxis_title="Maximum Wind Speed",
    yaxis_title="Number of Rides"
)

fig.show()

Rides and Precipitation

In [20]:
fig = px.scatter(
    bike_weather_daily,
    x="precipitation_sum",
    y="number_of_rides",
    trendline="ols",
    title="Daily Rides vs Precipitation"
)

fig.update_layout(
    xaxis_title="Daily Precipitation",
    yaxis_title="Number of Rides"
)

fig.show()

Rides vs temperature

In [21]:
import plotly.graph_objects as go


fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=bike_weather_daily["date"],
        y=bike_weather_daily["number_of_rides"],
        mode="lines",
        name="Daily Rides",
        yaxis="y1"
    )
)

fig.add_trace(
    go.Scatter(
        x=bike_weather_daily["date"],
        y=bike_weather_daily["temperature_2m_mean"],
        mode="lines",
        name="Daily Average Temperature",
        yaxis="y2"
    )
)

fig.update_layout(
    title="Daily Rides and Daily Average Temperature",
    xaxis=dict(
        title="Day"
    ),
    yaxis=dict(
        title="Daily Rides",
        side="left"
    ),
    yaxis2=dict(
        title="Daily Average Temperature",
        overlaying="y",
        side="right"
    ),
    hovermode="x unified",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

fig.show()